In [ ]:
using BSON, Dates, DelimitedFiles, Downloads, CUDA, cuDNN, Flux, Printf, Plots, JLD2
using Flux.Zygote
include("traindata.jl")
include("neural.jl");

In [ ]:
ρ_profiles, c1_profiles, ϕ_profiles, r_profiles = read_sim_data("tp_data.jld2") #radial inhomogenous systems, test particle density profiles, see Eq. (7)
ρ_windows_rad, c1_values_rad, ϕ_functions_rad, Rr_vals_rad = generate_inout_tp(ρ_profiles, c1_profiles, ϕ_profiles, r_profiles); 
#input: preprocessed density windows fρ (ρ_windows), pair potential(ϕ_functions), normalized radial coordinate(Rr_vals)
#output: c1-values(c1_values)

In [ ]:
ρ_profiles_c, c1_profiles_c, ϕ_profiles_c, r_profiles_c = read_sim_data_const("tp_data.jld2") #constant density
ρ_windows_c, c1_values_c, ϕ_functions_c, Rr_vals_c = generate_inout_tp(ρ_profiles_c, c1_profiles_c, ϕ_profiles_c, r_profiles_c);

In [ ]:
ρ_profiles_plan = load("plan_data.jld2")["ρ_profiles_plan"] #planar density profiles 
c1_profiles_plan = load("plan_data.jld2")["c1_profiles_plan"] #Eq. (9)
ϕ_profiles_plan =  load("plan_data.jld2")["ϕ_profiles_plan"]
r_plan = rand([-1, 1], length(ρ_profiles_plan)); # R(r) = +1 or -1 in the planar limit
ρ_windows_plan, c1_values_plan, ϕ_functions_plan, Rr_vals_plan = generate_inout_plan(ρ_profiles_plan, c1_profiles_plan, ϕ_profiles_plan,r_plan);

In [ ]:
ρ_windows = hcat(ρ_windows_plan,ρ_windows_c,ρ_windows_rad) #all training data
c1_values = hcat(c1_values_plan,c1_values_c,c1_values_rad)
ϕ_functions = hcat(ϕ_functions_plan,ϕ_functions_c,ϕ_functions_rad)
Rr_vals = hcat(Rr_vals_plan,Rr_vals_c,Rr_vals_rad);

In [ ]:
traindata = vcat(ρ_windows, ϕ_functions, Rr_vals); #all training input
traindata = Array(traindata)
c1_values = Array(c1_values); #training output 

In [ ]:
using CUDA
#set up model 
model = Chain(
    Dense(size(traindata)[1] => 512, softplus),
    Dense(512 => 256, softplus),
    Dense(256 => 128, softplus),
    Dense(128 => 64, softplus),
    Dense(64 => 32, softplus),
    Dense(32 => 1) 
) |> gpu

display(model)  

opt = Flux.setup(Adam(), model) 

loader = Flux.DataLoader((traindata, c1_values), batchsize=256, shuffle=true)  

loss(m, x, y) = Flux.mse(m(x), y)  
metric(m, x, y) = Flux.mae(m(x), y)  
get_learning_rate(epoch; initial=0.0001, rate=0.03, wait=5) = epoch < wait ? initial : initial * (1 - rate)^(epoch - wait)

model_savefile = "model_meta(new).bson"
println("Saving model to $(model_savefile)")
steps = 250
for epoch in 1:steps 
    learning_rate = get_learning_rate(epoch)
    Flux.adjust!(opt, learning_rate) 
    Flux.train!(loss, model, loader |> gpu, opt)
    println("completed: step $(epoch) of $(steps)")
    BSON.@save model_savefile model=cpu(model)
end

model